<a href="https://colab.research.google.com/github/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/blob/main/Notebooks/Ingresos_turisticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Cargar archivo
df_ingresos = pd.read_csv("https://raw.githubusercontent.com/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/refs/heads/main/Data/fact_ingresos_turisticos.csv")

print("Registros originales:", len(df_ingresos))

df_ingresos.head()

Registros originales: 48


,anio,mes,turistas,ingreso_usd
0,2021,1,193676,75034127
1,2021,2,163338,135985231
2,2021,3,282324,96732112
3,2021,4,91653,296250754
4,2021,5,124979,64420115


Revisar estructura

In [2]:
df_ingresos.info()

df_ingresos.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   anio         48 non-null     int64
 1   mes          48 non-null     int64
 2   turistas     48 non-null     int64
 3   ingreso_usd  48 non-null     int64
dtypes: int64(4)
memory usage: 1.6 KB


,0
anio,0
mes,0
turistas,0
ingreso_usd,0


**Transformar**

Eliminar duplicados

In [3]:
print(
    "Duplicados:",
    df_ingresos.duplicated().sum()
)

df_ingresos.drop_duplicates(inplace=True)

Duplicados: 0


Validar año

In [4]:
df_ingresos = df_ingresos[
    (df_ingresos['anio'] >= 2021)
    &
    (df_ingresos['anio'] <= 2024)
]

Validar mes

In [5]:
df_ingresos = df_ingresos[
    (df_ingresos['mes'] >= 1)
    &
    (df_ingresos['mes'] <= 12)
]

Validar ingresos

In [6]:
df_ingresos = df_ingresos[
    df_ingresos['ingreso_usd'] > 0
]

Validar turistas

In [7]:
df_ingresos = df_ingresos[
    df_ingresos['turistas'] > 0
]

Crear fecha analítica

In [8]:
df_ingresos['fecha'] = pd.to_datetime(
    df_ingresos['anio'].astype(str)
    + '-'
    + df_ingresos['mes'].astype(str)
    + '-01'
)

Crear trimestre

In [9]:
df_ingresos['trimestre'] = (
    df_ingresos['fecha']
    .dt.quarter
)

KPI: Gasto promedio por turista

In [10]:
df_ingresos['gasto_promedio_turista'] = (
    df_ingresos['ingreso_usd']
    /
    df_ingresos['turistas']
).round(2)

KPI: Ingreso en millones

In [11]:
df_ingresos['ingreso_millones'] = (
    df_ingresos['ingreso_usd']
    / 1000000
).round(2)

KPI: Clasificación de ingresos

In [12]:
df_ingresos['nivel_ingreso'] = np.where(
    df_ingresos['ingreso_usd'] < 50000000,
    'Bajo',
    np.where(
        df_ingresos['ingreso_usd'] < 100000000,
        'Medio',
        'Alto'
    )
)

Validación

In [13]:
print(df_ingresos.info())

df_ingresos.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   anio                    48 non-null     int64         
 1   mes                     48 non-null     int64         
 2   turistas                48 non-null     int64         
 3   ingreso_usd             48 non-null     int64         
 4   fecha                   48 non-null     datetime64[ns]
 5   trimestre               48 non-null     int32         
 6   gasto_promedio_turista  48 non-null     float64       
 7   ingreso_millones        48 non-null     float64       
 8   nivel_ingreso           48 non-null     object        
dtypes: datetime64[ns](1), float64(2), int32(1), int64(4), object(1)
memory usage: 3.3+ KB
None


,anio,mes,turistas,ingreso_usd,fecha,trimestre,gasto_promedio_turista,ingreso_millones
count,48.000000,48.000000,48.000000,4.800000e+01,48,48.000000,48.000000,48.000000
mean,2022.500000,6.500000,184126.229167,1.543948e+08,2022-12-16 05:00:00,2.500000,1079.212917,154.394583
min,2021.000000,1.000000,54163.000000,2.474667e+07,2021-01-01 00:00:00,1.000000,85.950000,24.750000
25%,2021.750000,3.750000,119247.250000,8.676735e+07,2021-12-24 06:00:00,1.750000,497.225000,86.767500
50%,2022.500000,6.500000,194188.000000,1.492490e+08,2022-12-16 12:00:00,2.500000,966.445000,149.245000
75%,2023.250000,9.250000,248150.500000,2.168580e+08,2023-12-08 18:00:00,3.250000,1202.282500,216.860000
max,2024.000000,12.000000,291712.000000,2.979803e+08,2024-12-01 00:00:00,4.000000,5084.100000,297.980000
std,1.129865,3.488583,72710.520234,8.221488e+07,NaN,1.129865,971.194791,82.214947


**Carga**

In [14]:
df_ingresos.to_csv(
    'dw_fact_ingresos_turisticos.csv',
    index=False
)

print("ETL completado correctamente")

ETL completado correctamente
